# Berkshire Hathaway Sentiment Analysis

This notebook loads precomputed sentiment features, merges in SPY yearly returns, builds sentiment regimes, and evaluates whether sentiment has signal for the following year.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from src.config.paths import FEATURES_DIR

features_path = FEATURES_DIR / "sentiment_features.csv"
print(f"Loading sentiment data from: {features_path}")

if not features_path.exists():
    raise FileNotFoundError(f"Sentiment file not found at {features_path}")

df = pd.read_csv(features_path)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)
df = df.sort_values("year").reset_index(drop=True)

spy = yf.download("SPY", start="1995-01-01", progress=False)

spy["year"] = spy.index.year

yearly_returns = (
    spy.groupby("year")["Adj Close"]
    .last()
    .pct_change()
    .rename("spy_return")
    .reset_index()
)

df = df.merge(yearly_returns, on="year", how="left")
df.head()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["year"], df["sentiment_score"], marker="o")
plt.title("Berkshire Hathaway Sentiment Over Time")
plt.xlabel("Year")
plt.ylabel("Sentiment Score")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
df["sentiment_ma"] = df["sentiment_score"].rolling(3).mean()
df["sentiment_regime"] = pd.cut(
    df["sentiment_score"],
    bins=[-1, -0.1, 0.1, 1],
    labels=["bearish", "neutral", "bullish"],
)

print(df["sentiment_regime"].value_counts())
df[["year", "sentiment_score", "sentiment_ma", "spy_return", "sentiment_regime"]].head()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["year"], df["sentiment_score"], label="Raw", marker="o")
plt.plot(df["year"], df["sentiment_ma"], label="3yr MA", marker="o")
plt.legend()
plt.title("Sentiment Score and 3-Year Moving Average")
plt.xlabel("Year")
plt.ylabel("Sentiment Score")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
plt.figure()
plt.scatter(df["sentiment_score"], df["spy_return"])
plt.title("Sentiment vs Next-Year SPY Returns")
plt.xlabel("Sentiment")
plt.ylabel("SPY Return")
plt.show()

df["next_year_return"] = df["spy_return"].shift(-1)
print(df.groupby("sentiment_regime")["next_year_return"].mean())

df_clean = df.dropna().copy()

df_clean["sentiment_regime"] = pd.qcut(
    df_clean["sentiment_score"], 3, labels=["low", "mid", "high"]
)

print(
    df_clean.groupby("sentiment_regime")["next_year_return"].mean()
)

print(df.nsmallest(5, "sentiment_score"))
